In [1]:
import anndata as ad
import numpy as np

In [2]:
a = ad.read_h5ad("../outputs/be1_all_scgpt.h5ad")
b = ad.read_h5ad("../outputs/be1_all_geneformer.h5ad")

In [3]:

import numpy as np
from scipy.stats import spearmanr
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

def compare_embeddings(X, Y, max_samples=5000, seed=42):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)

    assert X.ndim == Y.ndim == 2
    assert X.shape[0] == Y.shape[0]
    assert np.isfinite(X).all() and np.isfinite(Y).all()

    rng = np.random.default_rng(seed)

    # Stessi oggetti, stesso ordine
    n = X.shape[0]
    ids = rng.choice(n, min(n, max_samples), replace=False)
    X, Y = X[ids], Y[ids]

    if len(X) < 3:
        raise ValueError("Servono almeno 3 oggetti.")

    if np.any(np.linalg.norm(X, axis=1) == 0) or \
       np.any(np.linalg.norm(Y, axis=1) == 0):
        raise ValueError("Sono presenti embedding nulli.")

    # Normalizzazione L2 per ogni oggetto
    X = normalize(X)
    Y = normalize(Y)

    # 1. Linear CKA
    Xc = X - X.mean(axis=0, keepdims=True)
    Yc = Y - Y.mean(axis=0, keepdims=True)

    numerator = np.linalg.norm(Xc.T @ Yc, "fro") ** 2
    denominator = (
        np.linalg.norm(Xc.T @ Xc, "fro")
        * np.linalg.norm(Yc.T @ Yc, "fro")
    )
    cka = numerator / denominator if denominator > 0 else np.nan

    # 2. Similarity matrices
    Sx = X @ X.T
    Sy = Y @ Y.T

    # RSA Spearman - triangolo superiore, diagonale esclusa
    idx = np.triu_indices(len(X), k=1)
    rsa = spearmanr(Sx[idx], Sy[idx]).statistic

    # 3. k-NN overlap
    np.fill_diagonal(Sx, -np.inf)
    np.fill_diagonal(Sy, -np.inf)

    k_values = [k for k in [5, 10, 20, 50] if k < len(X)]
    overlap = {}

    for k in k_values:
        nx = np.argsort(-Sx, axis=1)[:, :k]
        ny = np.argsort(-Sy, axis=1)[:, :k]

        per_object = [
            len(set(a) & set(b)) / k
            for a, b in zip(nx, ny)
        ]
        overlap[k] = float(np.mean(per_object))

    # 4. PCA: dimensioni per spiegare il 95% della varianza
    def pca_stats(A):
        pca = PCA(svd_solver="full").fit(A)
        v = pca.explained_variance_ratio_
        dim95 = np.searchsorted(np.cumsum(v), 0.95) + 1

        p = v[v > 0]
        effective_rank = np.exp(-np.sum(p * np.log(p)))
        return int(dim95), float(effective_rank)

    dim_x, er_x = pca_stats(X)
    dim_y, er_y = pca_stats(Y)

    return {
        "n_objects": len(X),
        "linear_CKA": float(cka),
        "RSA_Spearman": float(rsa),
        "kNN_overlap": overlap,
        "PCA_95_dimensions": (dim_x, dim_y),
        "effective_rank": (er_x, er_y),
    }

In [4]:
# Utilizzo
X = a.obsm["X_scgpt_layer_12"]
Y = b.obsm["X_geneformer_layer_12"]

results = compare_embeddings(X, Y)

for name, value in results.items():
    print(f"{name}: {value}")

n_objects: 5000
linear_CKA: 0.7497803311103627
RSA_Spearman: 0.7782004277449631
kNN_overlap: {5: 0.14652, 10: 0.1897, 20: 0.2425, 50: 0.32097200000000004}
PCA_95_dimensions: (49, 173)
effective_rank: (22.917237049946795, 27.176232282796878)
